# SI4006 · M5 · Amparo — **Cuantizar el VLM** (S15, Parte A)

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 5

---

Sigue el lab de la profe (`sesiones/s15/S15_Lab_Despliegue_RESUELTO.ipynb`, **Parte A**): el mismo modelo cargado dos veces,
**fp16** y **int4** (bitsandbytes, nf4), y una tabla antes/después con memoria, latencia p50/p95 y calidad con temperatura 0.

- **Qué se cuantiza:** el VLM **Qwen2-VL-2B**, el único modelo que corre en la GPU del sistema que se despliega (M3 razona por Groq).
  Por qué este y no otro: `m5/README.md`, sección *Decisión*.
- **Calidad:** el harness de M4 (11 casos: `campos`, `valvula`, `exacto`, `juez_vlm`).
  - El juez de `juez_vlm` es siempre el VLM en fp16.
  - M3 comparte su caché entre las dos cargas: si int4 lee lo mismo, M3 responde lo mismo, y un cambio solo puede venir de la lectura.
- Al final se **precalcula el índice** de M3 para el Space (tabla de despliegue de la profe).

La lógica vive en `amparo/pipelines/m5_cuantizacion.py`; este notebook solo la llama.

> GPU (T4, L4 o A100) y `GROQ_API_KEY` en los secretos de Colab.

## 0 · Setup

In [1]:
%pip install -q "transformers>=4.45" accelerate bitsandbytes qwen-vl-utils pillow pandas groq sentence-transformers
# Colab trae torchao 0.10 y algunas librerias exigen >= 0.16 si lo encuentran; aqui no se usa.
%pip uninstall -q -y torchao
print('Listo.')

Listo.


In [2]:
# Clona el repo, o lo ACTUALIZA si ya estaba.
import os, sys, subprocess
from pathlib import Path

REPO_URL, RAMA = 'https://github.com/TomasPosada0626/Amparo.git', 'm5'
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'amparo' / 'nucleo' / 'schema.py').exists()), None)
if RAIZ is None:
    if Path('Amparo').is_dir():
        subprocess.run(['git', '-C', 'Amparo', 'pull', '-q', 'origin', RAMA], check=True)
    else:
        subprocess.run(['git', 'clone', '-q', '-b', RAMA, REPO_URL, 'Amparo'], check=True)
    RAIZ = Path('Amparo').resolve()
sys.path.insert(0, str(RAIZ))
print(subprocess.run(['git', '-C', str(RAIZ), 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CARPETA = '/content/drive/MyDrive/amparo_m5_cuantizacion'
except ImportError:
    CARPETA = None

from amparo.pipelines import m5_cuantizacion as m5q

6ee98fa docs: actualizar el grafo de graphify con S15 (cuantizacion y despliegue)

Mounted at /content/drive


## 1 · El contrato, antes de medir

Como en S14: *"si no lo escriben antes, después cualquier resultado parece bueno"*.
- **Latencia:** en GPU, int4 con bitsandbytes suele ir **igual o más lento** que fp16, porque descomprime los pesos en cada paso. Lo que se gana es **memoria**, así que la cláusula de latencia solo pide que no empeore más de un 30 %.
- **La válvula (m10)** no se negocia: el VLM no puede inventar la fecha ni el plazo que no se ven.

In [3]:
ctx = m5q.preparar(CARPETA)          # raiz, carpeta, clave de Groq, contrato y manifiesto (commit, GPU, versiones)

raiz: /content/Amparo · carpeta de trabajo: /content/drive/MyDrive/amparo_m5_cuantizacion
clave de Groq cargada desde los secretos de Colab
corrida: demo-final-13-g6ee98fa · NVIDIA L4
CONTRATO: {
 "componente": "VLM Qwen2-VL-2B-Instruct (lee el documento, M4)",
 "tecnica": "cuantizacion int4 (bitsandbytes nf4, computo fp16), la del lab de S15",
 "metrica": "harness de M4: 11 casos de m4/eval_set/casos_m4.csv",
 "memoria_x_min": 2.5,
 "campos_baja_max": 0.05,
 "valvula_min": 1.0,
 "exacto_no_peor": true,
 "latencia_peor_max": 0.3
}


## 2 · M3, una sola vez para las dos cargas

In [4]:
m5q.armar_m3(ctx);

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (6 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

M3 listo: 4 · agente + reglas


## 3 · Parte A · fp16 y int4, en la misma GPU

`cargar(cuantizar)` y `latencias()` son las del lab.
- **Memoria:** lo que reservó cada carga en la GPU.
- **Latencia:** 5 lecturas de la imagen de m01, más una de calentamiento que no cuenta.
- **Calidad:** el harness de M4 completo.

Primero se carga fp16, que se queda cargado como juez; luego int4.

In [5]:
medidas = m5q.medir(ctx)

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

... (26 outputs left out)

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

fp16: 4.42 GB · lectura p50 3.98 s · p95 3.99 s · campos 0.553 · exacto 1.0 · juez_vlm 1.0 · valvula 1.0


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

int4: 1.63 GB · lectura p50 5.58 s · p95 5.65 s · campos 0.543 · exacto 0.667 · juez_vlm 1.0 · valvula 1.0


## 4 · La tabla antes/después y el contrato

In [6]:
df = m5q.tabla(ctx)
display(df)

CONTRATO
  OK  memoria                  pedido: >= 2.5x menos              medido: 4.42 GB -> 1.63 GB (2.7x)
  OK  lectura (campos)         pedido: baja <= 0.05               medido: 0.553 -> 0.543 (baja 0.010)
  OK  valvula (m10)            pedido: >= 1.0                     medido: 1.0 -> 1.0
  NO  mecanismo (exacto)       pedido: no peor que fp16           medido: 3/3 -> 2/3
  NO  latencia de la lectura   pedido: p50 no mas de 30% peor     medido: 3.98 s -> 5.58 s (+40%)
  -> NO cumplido: la demo sigue con el VLM en fp16
  (juez_vlm: el juez es el VLM en fp16 en las dos filas)


,memoria_GB,lectura_p50_s,lectura_p95_s,campos,valvula,exacto,juez_vlm
modelo,,,,,,,
Qwen2-VL-2B · fp16,4.42,3.98,3.99,0.553,1.0,1.000,1.0
Qwen2-VL-2B · int4,1.63,5.58,5.65,0.543,1.0,0.667,1.0


### Lo que int4 leyó distinto

Campo por campo. Fechas, números y plazos son la debilidad conocida del VLM (README de M4): ahí es donde menos bits pueden pegar primero.

In [7]:
distintas = m5q.lecturas_distintas(ctx)

1 de 6 documentos leidos igual en int4 que en fp16
  doc01_respuesta_eps.png: pretension_visible 'La EPS NIEGA la autorización del procedimiento solicitado, porque no se encuentra contratado con ningún prestador de la red de servicios de su municipio de residencia.' -> 'La EPS NIEGA la autorización del procedimiento solicitado, porque no se encuentra contratado con ningún prestador de la red de servicios de su municipio de residencia. Le sugerimos consultar nuevamente cuando la red de prestadores sea actualizada.'
  doc02_oficio_planeacion.png: entidad None -> 'Alcaldía de Puerto Almendro'; pretension_visible 'Se le requiere por obra de construcción sin licencia.' -> 'Requerimiento por obra de construcción sin licencia'
  doc03_respuesta_cumplimiento.png: fecha '21 de septiembre de 2026' -> None; pretension_visible 'La Alcaldía cumplirá con el deber de publicar los contratos del municipio durante 2025 y 2026, y advirtió que, de no hacerlo, acudiría a la acción judicial correspondiente.

In [8]:
m5q.guardar(ctx)      # m5/cuantizacion.json en esta copia del repo y en Drive (entrega/)
m5q.descargar(ctx)    # baja el archivo: la copia de Colab se borra al cerrar

guardado:
  /content/Amparo/m5/cuantizacion.json
  /content/drive/MyDrive/amparo_m5_cuantizacion/entrega/cuantizacion.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

['/content/drive/MyDrive/amparo_m5_cuantizacion/entrega/cuantizacion.json']

## 5 · Para el despliegue: el índice de M3 precalculado

La tabla de despliegue de la profe pide *"precalculen el índice de embeddings; que no se construya al arrancar"*.
Se calcula una vez aquí y el archivo va al repo (`m5/space/indice_m3.npz`). El Space solo lo carga.

In [9]:
from amparo import despliegue as dsp
ruta = dsp.precalcular_indice()
import shutil; shutil.copy(ruta, Path(ctx.carpeta) / 'entrega' / ruta.name)
try:
    from google.colab import files; files.download(str(ruta))
except ImportError:
    pass

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

indice precalculado: 77 fragmentos · intfloat/multilingual-e5-base -> /content/Amparo/m5/space/indice_m3.npz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Lo que va a M5

- **La tabla antes/después** y el veredicto del contrato: `m5/cuantizacion.json` y el README de M5.
- **Si el contrato se cumple**, la demo en vivo (Kaggle T4) usa el VLM en int4 (`AMPARO_VLM=auto`). Si no se cumple, va en fp16 (`AMPARO_VLM=fp16`).
- **`m5/space/indice_m3.npz`** va al repo para el Space.

*Amparo · M5 · S15 · Parte A.*